# Suite ERE — The POSIX ERE dialect

`Expressions.POSIX.ERE` models POSIX's extended regular expressions (IEEE Std 1003.1, XBD 9.4, its grammar 9.5.3), as
grep reads its patterns with `-E`, a newline-separated list of them, and awk its EREs, with C's escapes, and as C++'s
`std::regex` reads them under its `extended`, `awk` and `egrep` grammars. Its kinds are validated as its grammar has
them, bind nothing, cover POSIX.1's four editions, print as an ERE writes them, or awk, are refused by evaluation by
name, and are checked complete against XBD 9.5 kept as data.

In [ ]:
import json
from pathlib import Path

from mbse.Expressions import Framework
from mbse.Expressions.POSIX import BRE, ERE
from mbse.Expressions.POSIX.BRE import Expressions as B
from mbse.Expressions.POSIX.ERE import Expressions as R, Text as RText
from mbse.Expressions.Framework import Evaluators as FE, Standards as FS, Symbolics as SY, Terms as F
from mbse.Schemas.Framework import Builders, JSON, Validators
from support import raises

store = Builders.Store()  # the POSIX family's dialects, ERE's install first: the installs run in any order
for install in [Framework.install, ERE.install, BRE.install]:
    store.transaction(install)
store.transaction(ERE.define)  # its editions


def term(spec, dialect=R.DIALECT):
    """The term value `spec` writes, the root of `dialect`'s holder, written in a transaction."""
    with store.transaction():
        family, own = dialect.path.split(".")
        return F.term(getattr(getattr(store.Expressions, family), own).Expression(lambda e: e.root(dialect.spec(spec))))


validate = lambda spec: R.DIALECT.validate(term(spec))  # noqa: E731
text = lambda spec, awk=False: RText.text(term(spec), awk)  # noqa: E731
union = lambda name: [b.name for b in store.registered(name).branches]  # noqa: E731: a union's branches
char = lambda c, spelling=None: lambda u: u.character(  # noqa: E731: a character, of its code point, and its spelling
    lambda r: r.value(ord(c)) if spelling is None else r.raw(spelling).value(ord(c)))
alt = lambda *elements: lambda u: u.alternative(lambda r: r.elements(list(elements)))  # noqa: E731
pattern = lambda *alternatives: lambda u: u.pattern(lambda r: r.alternatives(list(alternatives)))  # noqa: E731
capture = lambda *alternatives: lambda u: u.capturinggroup(lambda r: r.alternatives(list(alternatives)))  # noqa: E731
def repeat(element, least, most=None, greedy=None, spelling=None):
    """A quantifier: `element{least,most}`, lazy where not `greedy`, as `spelling` writes it where given."""
    def write(r):
        r = r.min(least).element(element)
        r = r if most is None else r.max(most)
        r = r if greedy is None else r.greedy(greedy)
        return r if spelling is None else r.raw(spelling)
    return lambda u: u.quantifier(write)


cls = lambda *elements, negate=False: lambda u: u.characterclass(lambda r: r.negate(negate).elements(list(elements)))  # noqa: E731
span = lambda low, high: lambda u: u.characterclassrange(lambda r: r.min(low).max(high))  # noqa: E731
chars = lambda kind: lambda u: u.characterset(lambda r: r.setKind(kind))  # noqa: E731
dot = chars("any")
anchor = lambda kind: lambda u: u.assertion(lambda r: r.assertionKind(kind))  # noqa: E731
named = lambda kind, name: lambda u: getattr(u, kind)(lambda r: r.name(name))  # noqa: E731: `[:alpha:]`, `[.a.]`, `[=a=]`
patterns = lambda *specs: lambda u: u.patternlist(lambda r: r.patterns(list(specs)))  # noqa: E731
error = lambda source: lambda u: u.error(lambda r: r.text(source))  # noqa: E731
missing = lambda kind: lambda u: u.missing(lambda r: r.kind(kind))  # noqa: E731
abc = pattern(alt(char("a"), char("b")), alt(char("c")))  # `ab|c`

<a id="ERE-01"></a>
## ERE-01 · Its union and holder: its kinds and invalid syntax, no backreference, a branch of the framework's terms, reading its family's other dialect; a holder refining the framework's, round-tripped through JSON

In [ ]:
assert R.TAGS == ("patternlist", "pattern", "alternative", "capturinggroup", "quantifier", "characterclass",
                  "characterclassrange", "characterset", "character", "assertion", "exclass", "collatingelement",
                  "equivalence") and R.HELPERS == () and R.SINCE == {}
assert list(R.DIALECT.kinds()) == union(R.DIALECT.union) == [*R.TAGS, "error", "missing"]
assert R.DIALECT.name() == "POSIX ERE" and R.DIALECT.tag == "posix_ere" and R.DIALECT.union == "Expressions.POSIX.ERE.Term"
assert "posix_ere" in union("Expressions.Framework.Term") and "posix_ere" in union("Expressions.Framework.Dialect")
assert R.DIALECT.reads() == (B.DIALECT,)  # its family's other dialect
Holder = store.registered("Expressions.POSIX.ERE.Expression")
assert Holder.ref and Holder.refines is store.registered("Expressions.Framework.Expression") and F.is_expression(Holder)
assert Holder.properties["root"].schema is store.registered(R.DIALECT.union) and list(Holder.adjacencies) == ["abi"]
with store.transaction():
    holder = store.Expressions.POSIX.ERE.Expression(lambda e: e.root(R.DIALECT.spec(abc)))
assert Validators.Validate(store)(Holder, holder) == [] and R.DIALECT.validate(holder) == []
assert F.dialect_of(F.term(holder)) == R.DIALECT.union and F.kind_of(F.term(holder))[0] == "pattern"
assert F.same(JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, holder)).root, holder.root)
with raises(Exception, match="'backreference' is not a member of a union value"):  # an ERE has no backreference (9.5.1)
    with store.transaction():
        store.Expressions.POSIX.ERE.Expression(lambda e: e.root(R.DIALECT.spec(lambda u: u.backreference(lambda r: r.ref(1)))))
assert R.DIALECT.validate(term(lambda u: u.backreference(lambda r: r.ref(1)), B.DIALECT)) == []  # a BRE's, as it reads it

<a id="ERE-02"></a>
## ERE-02 · Its kinds: their properties in order, each term property its Term, a quantifier greedy or not; forms and make round-trip

In [ ]:
props = lambda name: list(store.registered(f"Expressions.POSIX.ERE.{name}").properties)  # noqa: E731
assert props("PatternList") == ["patterns"] and props("Pattern") == props("CapturingGroup") == ["alternatives"]
assert props("Alternative") == ["elements"] and props("Quantifier") == ["raw", "min", "max", "greedy", "element"]
assert props("CharacterClass") == ["negate", "elements"] and props("CharacterClassRange") == ["min", "max"]
assert props("CharacterSet") == ["setKind"] and props("Character") == ["raw", "value"] and props("Assertion") == ["assertionKind"]
assert props("ExClass") == props("CollatingElement") == props("Equivalence") == ["name"]
prop = lambda name, member: store.registered(f"Expressions.POSIX.ERE.{name}").properties[member].schema  # noqa: E731
Term = store.registered(R.DIALECT.union)
assert prop("Quantifier", "element") is prop("CharacterClassRange", "min") is prop("CharacterClassRange", "max") is Term
assert prop("PatternList", "patterns").element is prop("Pattern", "alternatives").element is prop("Alternative", "elements").element is Term
quantified = F.form(term(repeat(cls(span(char("a"), char("z"))), 1, greedy=False)))
assert quantified.attributes == {"min": 1, "greedy": False} and [F.kind_of(a)[0] for a in quantified.arguments] == ["characterclass"]
for spec in [patterns(abc, pattern(alt(dot, anchor("end")))), repeat(repeat(anchor("start"), 1), 0, 1),
             cls(named("exclass", "alpha"), named("collatingelement", "ch"), named("equivalence", "e"), char("-"), negate=True)]:
    assert F.same(term(R.DIALECT.make(F.form(term(spec)))), term(spec))

<a id="ERE-03"></a>
## ERE-03 · Its checks: what each kind's terms may be, as XBD 9.5.3 has them, an anchor anywhere, and its attributes' values

In [ ]:
for spec, problems in [
    (patterns(alt(char("a"))), ["a patternlist's patterns cannot hold an alternative"]),
    (pattern(char("a")), ["a pattern's alternatives cannot hold a character"]),
    (alt(pattern(alt())), ["an alternative's elements cannot hold a pattern"]),
    (alt(named("exclass", "alpha")), ["an alternative's elements cannot hold an exclass"]),
    (capture(char("a")), ["a capturinggroup's alternatives cannot hold a character"]),
    (repeat(alt(), 0), ["a quantifier's element cannot hold an alternative"]),
    (cls(dot), ["a characterclass's elements cannot hold a characterset"]),  # `.` within brackets is a character
    (span(dot, named("collatingelement", "z")), ["a characterclassrange's min cannot hold a characterset"]),
    (repeat(char("a"), -1), ["a quantifier's min cannot be negative, got -1"]),
    (repeat(char("a"), 3, 2), ["a quantifier's max cannot be less than its min, got 2 and 3"]),
    (lambda u: u.quantifier(lambda r: r.min(1)), ["a quantifier needs an element"]),
    (chars("word"), ["CharacterSet.setKind cannot be 'word'"]),
    (lambda u: u.character(lambda r: r.value(-1)), ["a character's value must be a code point, got -1"]),
    (anchor("lookahead"), ["Assertion.assertionKind cannot be 'lookahead'"]),
    (named("equivalence", "=e"), ["an equivalence's name cannot hold '='"]),
]:
    assert validate(spec) == problems, (validate(spec), problems)
for spec in [abc, alt(char("a"), anchor("start"), anchor("end"), char("b")),  # `a^$b`: an anchor anywhere (ERE_expression)
             capture(alt(anchor("start"), char("a"))), repeat(repeat(char("a"), 1), 0, 1),  # `ERE_expression ERE_dupl_symbol`
             repeat(anchor("start"), 0), repeat(char("a"), 0, greedy=False)]:  # POSIX.1-2024's repetition modifier
    assert validate(spec) == [], spec

<a id="ERE-04"></a>
## ERE-04 · grep -E's list of patterns, a line each; awk's EREs: their characters printed with C's escapes

In [ ]:
listed = patterns(pattern(alt(anchor("start"), char("a"))), abc)
assert validate(listed) == [] and text(listed) == "^a\nab|c"  # grep's pattern_list with -E (XCU grep)
awk = pattern(alt(char("\n"), char("\x07"), char("\x08"), char("\x01"), char("\x7f"), char("/"), char("+"),
                  cls(char("\t"), char("+"))))
assert text(awk, True) == "\\n\\a\\b\\001\\177/\\+[\\t+]"  # XCU awk, Regular Expressions: C's escapes, in brackets too
assert text(awk) == "\n\x07\x08\x01\x7f/\\+[\t+]" == RText.ToText(term(awk))  # an ERE writes a control character as itself
assert text(pattern(alt(char("\n", "\\012"))), True) == "\\012"  # a spelling as written

<a id="ERE-05"></a>
## ERE-05 · What it binds: nothing; a subexpression numbered, which no backreference resolves

In [ ]:
assert not any(F.is_binder(b.schema) for b in store.registered(R.DIALECT.union).branches)
grouped = term(pattern(alt(capture(alt(char("a"))), repeat(capture(alt(char("b"))), 0))))
assert SY.free(grouped, R.DIALECT) == set() and R.DIALECT.validate(grouped) == []
assert [F.kind_of(t)[0] for t in F.walk(grouped)] == ["pattern", "alternative", "capturinggroup", "alternative",
                                                      "character", "quantifier", "capturinggroup", "alternative", "character"]

<a id="ERE-06"></a>
## ERE-06 · Invalid syntax: error and missing in its Term, validated, printed as written, refused by name

In [ ]:
assert validate(error("(")) == ["invalid syntax '('"] and text(error("(")) == "("
assert validate(missing("atom")) == ["missing atom"] and text(alt(char("a"), missing("atom"))) == "a"
assert validate(pattern(alt(char("a"), error("{")))) == ["argument 0: argument 1: invalid syntax '{'"]
with raises(ValueError, match="POSIX ERE error has no text to print"):
    text(lambda u: u.error(lambda r: r))

<a id="ERE-07"></a>
## ERE-07 · Its editions: POSIX.1's four, as POSIX BRE's, each with every kind

In [ ]:
ere_ = store.Instances.Expressions.POSIX.ERE
assert [FS.designation_of(s) for s in FS.standards(ere_).values()] == [
    "IEEE Std 1003.1-2001", "IEEE Std 1003.1-2008", "IEEE Std 1003.1-2017", "IEEE Std 1003.1-2024"]
assert R.EDITIONS == B.EDITIONS and FS.find(ere_, "ISO/IEC 9945:2003") is FS.find(ere_, "POSIX.1-2001")  # C++17's and C++20's
assert all([s.name for s in FS.kinds(e)] == [f"Expressions.POSIX.ERE.{name}" for name in (
    "PatternList", "Pattern", "Alternative", "CapturingGroup", "Quantifier", "CharacterClass", "CharacterClassRange",
    "CharacterSet", "Character", "Assertion", "ExClass", "CollatingElement", "Equivalence")]
           for e in FS.standards(ere_).values())
assert store.transaction(ERE.define)[0]["POSIX_2024"] is FS.find(ere_, "POSIX.1-2024")  # run again, it writes nothing

<a id="ERE-08"></a>
## ERE-08 · Printing: every kind as an ERE writes it, its special characters escaped, a lazy quantifier, a repeated one unambiguous; a BRE's term as a BRE

In [ ]:
every = pattern(alt(anchor("start"), capture(alt(char("a")), alt(char("."))), repeat(char("b"), 0), repeat(char("c"), 1),
                    repeat(char("d"), 0, 1), repeat(char("e"), 2, 2), repeat(char("f"), 2), repeat(char("g"), 2, 5),
                    cls(span(char("a"), char("z")), named("exclass", "digit"), named("collatingelement", "ch"),
                        named("equivalence", "e"), char("]"), negate=True), dot, anchor("end")))
assert text(every) == "^(a|\\.)b*c+d?e{2}f{2,}g{2,5}[^a-z[:digit:][.ch.][=e=]]].$"
assert text(pattern(alt(repeat(char("x"), 0, greedy=False), repeat(char("y"), 1, 3, greedy=False)))) == "x*?y{1,3}?"
assert text(pattern(alt(repeat(repeat(char("a"), 1), 0, 1), repeat(anchor("start"), 0)))) == "a+{0,1}^*"  # never `a+?`
assert text(pattern(alt(*(char(c) for c in "^.[$()|*+?{\\"), cls(char("("), char("\\"))))) == "\\^\\.\\[\\$\\(\\)\\|\\*\\+\\?\\{\\\\[(\\]"
assert text(repeat(char("a"), 0, spelling="{0,}")) == "a{0,}" and text(char("A", "\\A")) == "\\A"  # as written
assert RText.ToText(term(pattern(alt(repeat(char("a"), 1), lambda u: u.backreference(lambda r: r.ref(1)))), B.DIALECT)) == "a\\{1,\\}\\1"

<a id="ERE-09"></a>
## ERE-09 · Evaluation: each kind refused by its dialect's name

In [ ]:
interpreter = FE.Interpreter(R.DIALECT, {}, evaluated=frozenset())  # no kind evaluated, each refused by its dialect's name
for spec in [patterns(abc), abc, char("a"), cls(), anchor("end")]:
    with raises(NotImplementedError, match=f"POSIX ERE {F.kind_of(term(spec))[0]} is not evaluated"):
        interpreter(term(spec))
with raises(ValueError, match="POSIX ERE error is not evaluated"):
    interpreter(term(error("(")))

<a id="ERE-10"></a>
## ERE-10 · Completeness: grammars/posix-regex.json maps every kind, awk's escapes and POSIX.1-2024's repetition modifier among its entries

In [ ]:
grammar = json.loads(Path("../../grammars/posix-regex.json").read_text(encoding="utf-8"))
nodes = grammar["nodes"]
mapped = {kind for n in nodes for kind in n.get("kinds", ())}
assert {kind for kind in mapped if kind.startswith("ERE.")} == {f"ERE.{tag}" for tag in R.TAGS}  # every kind, exactly
assert mapped == {f"BRE.{tag}" for tag in B.TAGS} | {f"ERE.{tag}" for tag in R.TAGS}  # and no other dialect's
by = {n["node"]: n for n in nodes}
assert by["escape sequence"]["kinds"] == ["ERE.character"] and by["escape sequence"]["source"].endswith("XCU awk, Regular Expressions")
assert "not" in by["repetition modifier"] and by["repetition modifier"]["source"] == "IEEE Std 1003.1-2024, XBD 9.4.6"
assert by["ERE_expression"]["kinds"] == ["ERE.assertion", "ERE.capturinggroup", "ERE.quantifier"]